# ViDoRe V3 Pharmaceuticals subset

## 1. Load the four subsets

In [1]:
!nvidia-smi

Wed Oct  7 01:43:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:E1:00.0 Off |                    0 |
| 30%   35C    P8             19W /  450W |       1MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from collections import Counter, defaultdict
from pprint import pprint

from datasets import load_dataset

REPO = "vidore/vidore_v3_pharmaceuticals"
corpus  = load_dataset(REPO, "corpus")["test"]
docs    = load_dataset(REPO, "documents_metadata")["test"]
queries = load_dataset(REPO, "queries")["test"]
qrels   = load_dataset(REPO, "qrels")["test"]

print("------corpus-----")
pprint(corpus)
print("------example-record")
pprint(corpus[0])
# # Keys: corpus_id, doc_id, page_number_in_doc
# # Content: image, markdown

# print("-------Docs------")
pprint(docs)
print("------example-record")
pprint(docs[0])
# # keys: doc_id / filename, page_no.
# # content: I feel page number is useful, visual types can be helpful to understand what type of data, doc language, 
print("-------queries---")
pprint(queries)
pprint(queries[0])
# keys: query_id, 
# content: query, language, raw_answers, answer
print("------qrels------")
pprint(qrels)
pprint(qrels[0])
# keys: query_id, corpus_id,
# content: score

/storage/homefs/kr23w045/vdoc-agent/envs/train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


------corpus-----
Dataset({
    features: ['corpus_id', 'image', 'doc_id', 'markdown', 'page_number_in_doc'],
    num_rows: 2313
})
------example-record
{'corpus_id': 0,
 'doc_id': 'final_ce_dental_caries',
 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=2000x1125 at 0x7F473ED2C680>,
 'markdown': 'U.S. FOOD& DRUG ADMINISTRATION\n'
             'FDA\n'
             'FDA Drug Topics:\n'
             'An Update on Transmucosal\n'
             'Buprenorphine and Dental Caries\n'
             'Mark A. Liberatore, PharmD, RAC\n'
             'Commander, United States Public Health Service\n'
             'Deputy Director for Safety\n'
             'Division of Anesthe Addiction Medicine, and Pain Medicine\n'
             'Office of New Drugs, Center for Drug Evaluation and Research,\n'
             'U.S. Food and Drug Administration',
 'page_number_in_doc': 0}
Dataset({
    features: ['file_name', 'doc_id', 'url', 'doc_type', 'doc_language', 'doc_year', 'visual_types', 'page_n

## 2. English queries only

Each question exists in 6 languages and we first work on the ones in English language.

In [3]:
print("length of all queries", len(queries))
print("unique languages present in the queries: ", queries.unique('language'))
# lets get just the english queries
english = queries.filter(lambda lang: lang == "english", input_columns="language")
english_ids = set(english["query_id"])
print("english queries:", len(english_ids))

length of all queries 2184
unique languages present in the queries:  ['english', 'french', 'spanish', 'italian', 'german', 'portuguese']
english queries: 364


## 3. Map queries to documents

What is the granularity here, ??? 
1 row in corpus is 1 page, contains images, markdown etc. . 
1 row in documents_metadata is 1 PDF document, contains file metadata - file_name, url, doc_type, so on so on.

Now we need to find out the question's answer is in corpus_id and doc_id
The queries table does not say, so follow the chain through the other two tables:

```
question  --(qrels)-->  page  --(corpus)-->  document
query_id                corpus_id            doc_id
```

The key is `corpus_id` (unique across the corpus)

In [4]:
# key of this dict should be page number and the value should be the document it belongs to.
page_to_doc = dict(zip(corpus["corpus_id"], corpus["doc_id"]))
print("pages:", len(page_to_doc))
# pprint(page_to_doc)

query_to_docs = defaultdict(set)
for qid, cid in zip(qrels["query_id"], qrels["corpus_id"]):
    if qid in english_ids:  # skip the translated copies
        query_to_docs[qid].add(page_to_doc[cid])

print("english queries with at least one page:", len(query_to_docs))
multi = {q: d for q, d in query_to_docs.items() if len(d) > 1}
print("queries spanning more than one document:", len(multi))

pages: 2313
english queries with at least one page: 364
queries spanning more than one document: 50


## 4. Group documents that share a question

If a question uses documents A and B, they must land on the **same side** of the split.
So we split *groups*: A shares a question with B, B with C → A, B, C are one group.

Each group is named after its alphabetically first document (`min`), so the names are the same on every run.

In [5]:
# Every document starts in its own group, named after itself
group_of = {d: d for d in page_to_doc.values()}

# A question that touches several groups merges them: rename them all to one name
for doc_set in query_to_docs.values():
    names = {group_of[d] for d in doc_set} # the groups this question touches
    keep = min(names)        # the name to keep (min: same on every run)
    for d in group_of:
        if group_of[d] in names:
            group_of[d] = keep

# group name -> its documents
groups = defaultdict(set)
for d, g in group_of.items():
    groups[g].add(d)

# questions per group (a question's documents all share one group, so ask any of them)
q_per_group = Counter(group_of[min(ds)] for ds in query_to_docs.values())

print("documents:", len(group_of), "-> groups:", len(groups))
print("docs in the 5 biggest groups:", sorted((len(g) for g in groups.values()), reverse=True)[:5])
print("questions in the 5 biggest groups:", [n for _, n in q_per_group.most_common(5)])


documents: 52 -> groups: 25
docs in the 5 biggest groups: [14, 9, 3, 3, 2]
questions in the 5 biggest groups: [70, 64, 32, 26, 23]


## 5. Split 30 / 70 by group

Every group gets a fixed, random-looking position from its name and `SEED`. Walk the groups in that order:
a group joins dev while dev stays at or under 30% of the questions; the rest is test.

Same data + same `SEED` → same split, every run. Change `SEED` or `DEV_FRACTION` to try another split.
This cell only computes; the next one saves.

In [6]:
import hashlib

SEED = 0
DEV_FRACTION = 0.30

def position(group_name):
    """A fixed, random-looking sort key: same name + same SEED -> same position."""
    return hashlib.sha256(f"{SEED}:{group_name}".encode()).hexdigest()

# questions in each group (all documents of a question share one group, so ask any of them)
questions_in = defaultdict(list)
for qid, doc_set in query_to_docs.items():
    questions_in[group_of[min(doc_set)]].append(qid)

# walk the groups in position order; a group joins dev while dev stays <= DEV_FRACTION
target = round(DEV_FRACTION * len(query_to_docs))
dev_ids = set()
for g in sorted(questions_in, key=position):
    if len(dev_ids) + len(questions_in[g]) <= target:
        dev_ids.update(questions_in[g])
test_ids = set(query_to_docs) - dev_ids

dev_docs = {d for q in dev_ids for d in query_to_docs[q]}
test_docs = {d for q in test_ids for d in query_to_docs[q]}
assert not dev_docs & test_docs, "a document is in both dev and test"

print(f"dev {len(dev_ids)} questions, {len(dev_docs)} docs | test {len(test_ids)} questions, {len(test_docs)} docs")

dev 108 questions, 13 docs | test 256 questions, 37 docs


## 6. Save the split (safe to re-run)

- Saved files already match → nothing happens.
- They differ → nothing is written; set `OVERWRITE = True` to replace them on purpose.

Once you start tuning on dev, keep the split fixed, and never open `test.jsonl` before P28.

In [7]:
import json
from pathlib import Path

OVERWRITE = False                     # True = replace the saved split with the one computed above

EVALS = Path("../evals") if Path("../evals").is_dir() else Path("evals")

# relevant pages per question, with their score (2 = fully, 1 = partly relevant)
pages_of = defaultdict(list)
for qid, cid, score in zip(qrels["query_id"], qrels["corpus_id"], qrels["score"]):
    if qid in english_ids:
        pages_of[qid].append({"corpus_id": cid, "score": score})

# one record per English question
fields = ["query_id", "query", "answer", "query_types", "content_type"]
record = {r["query_id"]: {**r, "doc_ids": sorted(query_to_docs[r["query_id"]]), "pages": pages_of[r["query_id"]]}
          for r in english.select_columns(fields)}

def jsonl(ids):
    return "".join(json.dumps(record[q]) + "\n" for q in sorted(ids))

files = {"dev.jsonl": jsonl(dev_ids), "test.jsonl": jsonl(test_ids)}
files["split.json"] = json.dumps({
    "dataset": REPO, "language": "english", "seed": SEED, "dev_fraction": DEV_FRACTION,
    "unit": "document groups: documents that share a question stay on the same side",
    "dev_questions": len(dev_ids), "test_questions": len(test_ids),
    "dev_docs": sorted(dev_docs), "test_docs": sorted(test_docs),
    "test_sha256": hashlib.sha256(files["test.jsonl"].encode()).hexdigest(),
}, indent=2) + "\n"

saved = {name: (EVALS / name).read_text() for name in files if (EVALS / name).exists()}
if saved == files:
    print("Saved split matches. Nothing to do.")
elif saved and not OVERWRITE:
    print("The saved split is different from this one. Nothing written.")
    print("Set OVERWRITE = True and re-run to replace it.")
else:
    for name, text in files.items():
        (EVALS / name).write_text(text)
    print("Saved:", ", ".join(str(EVALS / n) for n in files))

The saved split is different from this one. Nothing written.
Set OVERWRITE = True and re-run to replace it.
